In [1]:
from unsloth import FastLanguageModel
import json

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/ubuntu/miniconda/envs/unsloth_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.5.1 with CUDA 1201 (you have 2.8.0+cu128)
    Python  3.11.10 (you have 3.11.13)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


🦥 Unsloth Zoo will now patch everything to make training faster!


In [15]:
max_seq_length = None
load_in_4bit = True
dtype = None 
model_name_str = 'end_to_end_v5_with_cross_val'
model_name = f'/data2/finetuned_llms/{model_name_str}'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name, # YOUR MODEL YOU USED FOR TRAINING
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

from datasets import load_dataset

base_path = '/data2/jsonl/cross_val_and_train_eval.jsonl'

data_files = {
    'val': base_path
}

dataset = load_dataset(
    "json",
    data_files=data_files
)

print(dataset['val']['text'][0])


==((====))==  Unsloth 2025.8.1: Fast Llama patching. Transformers: 4.55.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.278 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Generating val split: 17600 examples [00:00, 369041.55 examples/s]

<|start_header_id|>user<|end_header_id|>

You are helping decode speech from neural activity to help restore communication for a paralyzed patient. For each time bin of neural activity, a neural network model provides the 10 most probable tokens. Tokens consist of ARPAbet phonemes and the space character, denoted as <>. On each line, the the top 10 tokens are listed in order, from most to least likely. Each separate line represents the model output for a given non-overlapping neural time bin, starting from the beginning of the text. Since the model output is not perfectly accurate, your job is to correct its output by producing the ground-truth phoneme sequence along with the corresponding ground-truth word-level sentence. Produce coherent text that is gramatically correct. Output only the corrected phoneme sequence and word-level sentence, no additional explanations or metadata.

K G S T L F HH D JH P
AA AO AY AE OW AH EY EH IH AW
F R S T L P V K <> Z
IY EY AH ER AY IH S F T L
<> Z S 

In [23]:
import numpy as np
batch_indices = np.concatenate((np.arange(100), np.arange(801,900)))

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 801, 802, 803, 804,
       805, 806, 807, 808, 809, 810, 811, 812, 813, 814, 815, 816, 817,
       818, 819, 820, 821, 822, 823, 824, 825, 826, 827, 828, 829, 830,
       831, 832, 833, 834, 835, 836, 837, 838, 839, 840, 841, 842, 843,
       844, 845, 846, 847, 848, 849, 850, 851, 852, 853, 854, 855, 856,
       857, 858, 859, 860, 861, 862, 863, 864, 865, 866, 867, 868, 869,
       870, 871, 872, 873, 874, 875, 876, 877, 878, 879, 880, 88

In [27]:
batch_size = 1
split = "val"

skip_even = False

last_lines_val = []
n = len(dataset[split])

for batch_idx in batch_indices:
    
    if batch_idx % 80 == 0:
        print(batch_idx)
        

    # Grab a batch of texts
    batch = dataset[split][int(batch_idx)]
    val_texts = batch["text"]
    
    # Tokenize the batch
    inputs = tokenizer(
        val_texts,
        return_tensors='pt',
        padding=True,
        truncation=True
    ).to('cuda')

    # Generate
    outputs = model.generate(
        **inputs,
        use_cache=True, 
        max_new_tokens=400
    )

    # Decode all outputs
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # Keep just the last non-empty line from each sequence
    for seq in decoded:
        lines = [l.strip() for l in seq.splitlines() if l.strip()]
        last_line = lines[-1] if lines else ""
        last_lines_val.append(last_line)
        

with open(f'/data2/jsonl/val_sents_{model_name_str}.json', "w") as f:
    json.dump(last_lines_val, f)

0
0 val
1 val
2 val
3 val
4 val
5 val


KeyboardInterrupt: 

In [26]:
print(dataset['val']['text'][10])

<|start_header_id|>user<|end_header_id|>

You are helping decode speech from neural activity to help restore communication for a paralyzed patient. For each time bin of neural activity, a neural network model provides the 10 most probable tokens. Tokens consist of ARPAbet phonemes and the space character, denoted as <>. On each line, the the top 10 tokens are listed in order, from most to least likely. Each separate line represents the model output for a given non-overlapping neural time bin, starting from the beginning of the text. Since the model output is not perfectly accurate, your job is to correct its output by producing the ground-truth phoneme sequence along with the corresponding ground-truth word-level sentence. Produce coherent text that is gramatically correct. Output only the corrected phoneme sequence and word-level sentence, no additional explanations or metadata.

AY AH <> AE EY IY OW HH IH M
<> AY V EY IY Z M K L AH
K <> G HH T S SH W CH AH
AA AO AH OW EH AY AE UH AW 

In [14]:
import json
from cer_wer import _cer_and_wer
with open("/data2/jsonl/val_ground_truth.json", "r") as f:
    val_gt = json.load(f)
    
cer, wer, _ = _cer_and_wer(last_lines_val, val_gt, returnCI=False)
print(wer)

0.27350427350427353
